# Chương 8: Thực hành - Học máy Cổ điển (Classical Machine Learning)

**Mục tiêu bài thực hành:**
1. **$k$-Nearest Neighbors ($k$-NN) từ đầu:** Cài đặt hàm tính khoảng cách, bầu chọn nhãn đa số và khảo sát ranh giới quyết định theo $k$.
2. **Cây quyết định (Decision Tree) từ đầu:** Tự viết hàm tính chỉ số Gini Impurity, thuật toán phân chia tốt nhất (Best Split) và đệ quy dựng cây phân loại.
3. **Hồi quy Logistic từ đầu:** Tối ưu hóa hàm mất mát Binary Cross-Entropy bằng Gradient Descent.
4. **Sức mạnh của Học tổ hợp (Ensemble Learning):** Xây dựng mô hình Random Forest từ đầu và đối chiếu trực tiếp ranh giới quyết định với Cây quyết định đơn lẻ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Thuật toán $k$-Láng giềng gần nhất ($k$-NN) từ đầu và Tác động của $k$

In [ ]:
# Sinh dữ liệu 2 cụm hình bán nguyệt (Two Moons)
def generate_moons(n_samples=200, noise=0.25):
    n_half = n_samples // 2
    t1 = np.linspace(0, np.pi, n_half)
    x1 = np.cos(t1) + np.random.normal(0, noise, n_half)
    y1 = np.sin(t1) + np.random.normal(0, noise, n_half)
    
    t2 = np.linspace(0, np.pi, n_half)
    x2 = 1 - np.cos(t2) + np.random.normal(0, noise, n_half)
    y2 = 0.5 - np.sin(t2) + np.random.normal(0, noise, n_half)
    
    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.array([0] * n_half + [1] * n_half)
    return X, y

X_data, y_data = generate_moons(n_samples=200, noise=0.22)

class KNNClassifierScratch:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        
    def predict(self, X_test):
        preds = []
        for x in X_test:
            # Tính khoảng cách Euclidean đến mọi điểm huấn luyện
            dists = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
            # Lấy chỉ số của k láng giềng gần nhất
            k_indices = np.argsort(dists)[:self.k]
            k_labels = self.y_train[k_indices]
            # Bầu chọn đa số
            counts = np.bincount(k_labels)
            preds.append(np.argmax(counts))
        return np.array(preds)

# Khảo sát ranh giới quyết định khi k = 1, 7, 35
x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
k_values = [(1, "k=1 (Overfitting, ranh giới góc cạnh)"), 
            (7, "k=7 (Cân bằng tối ưu)"), 
            (35, "k=35 (Underfitting, ranh giới quá mượt)")]

for idx, (k_val, title) in enumerate(k_values):
    knn = KNNClassifierScratch(k=k_val)
    knn.fit(X_data, y_data)
    Z = knn.predict(grid_pts).reshape(xx.shape)
    
    ax = axes[idx]
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], alpha=0.3, colors=['#3498db', '#e74c3c'])
    ax.scatter(X_data[y_data == 0, 0], X_data[y_data == 0, 1], color='#2980b9', edgecolors='k', s=30)
    ax.scatter(X_data[y_data == 1, 0], X_data[y_data == 1, 1], color='#c0392b', edgecolors='k', s=30)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle("Tác động của Siêu tham số k lên Ranh giới Quyết định của k-NN", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Cài đặt Cây quyết định (Decision Tree from scratch) bằng Chỉ số Gini

In [ ]:
def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    p1 = np.mean(y == 1)
    p0 = 1.0 - p1
    return 1.0 - (p0**2 + p1**2)

class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value # Nếu là nút lá, lưu trữ nhãn dự đoán
        
class SimpleDecisionTree:
    def __init__(self, max_depth=4, min_samples=4):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # Điều kiện dừng
        if depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples:
            leaf_value = 1 if np.mean(y) >= 0.5 else 0
            return TreeNode(value=leaf_value)
            
        # Tìm nhát cắt tối ưu làm giảm Gini nhiều nhất
        best_gini = 999.0
        best_feat, best_thresh = None, None
        
        for feat in range(n_features):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left_mask = X[:, feat] <= thresh
                right_mask = ~left_mask
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                g_left = gini_impurity(y[left_mask])
                g_right = gini_impurity(y[right_mask])
                weighted_gini = (np.sum(left_mask) * g_left + np.sum(right_mask) * g_right) / n_samples
                
                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feat = feat
                    best_thresh = thresh
                    
        if best_feat is None:
            return TreeNode(value=1 if np.mean(y) >= 0.5 else 0)
            
        left_sub = self._grow_tree(X[X[:, best_feat] <= best_thresh], y[X[:, best_feat] <= best_thresh], depth + 1)
        right_sub = self._grow_tree(X[X[:, best_feat] > best_thresh], y[X[:, best_feat] > best_thresh], depth + 1)
        return TreeNode(feature=best_feat, threshold=best_thresh, left=left_sub, right=right_sub)
        
    def predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left)
        return self.predict_one(x, node.right)
        
    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])

tree = SimpleDecisionTree(max_depth=5)
tree.fit(X_data, y_data)
acc_tree = np.mean(tree.predict(X_data) == y_data)
print(f"[Cây quyết định from scratch] Độ chính xác trên tập train: {acc_tree * 100:.2f}%")

## 3. Sức mạnh Học tổ hợp: Cây quyết định đơn lẻ vs. Rừng ngẫu nhiên (Random Forest)

Xây dựng mô hình Random Forest kết hợp $B = 25$ cây quyết định độc lập, mỗi cây được huấn luyện trên một mẫu Bootstrap có hoàn lại.

In [ ]:
class SimpleRandomForest:
    def __init__(self, n_estimators=25, max_depth=5):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.trees = []
        
    def fit(self, X, y):
        n_samples = X.shape[0]
        self.trees = []
        for _ in range(self.n_estimators):
            # Lấy mẫu Bootstrap có hoàn lại
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            tree_b = SimpleDecisionTree(max_depth=self.max_depth)
            tree_b.fit(X[indices], y[indices])
            self.trees.append(tree_b)
            
    def predict(self, X):
        # Biểu quyết đa số từ tất cả các cây
        all_preds = np.array([t.predict(X) for t in self.trees]) # n_estimators x N
        majority = (np.mean(all_preds, axis=0) >= 0.5).astype(int)
        return majority

rf = SimpleRandomForest(n_estimators=25, max_depth=5)
rf.fit(X_data, y_data)

# Trực quan hóa ranh giới so sánh Cây đơn lẻ vs Rừng ngẫu nhiên
Z_tree = tree.predict(grid_pts).reshape(xx.shape)
Z_rf = rf.predict(grid_pts).reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.contourf(xx, yy, Z_tree, levels=[-0.5, 0.5, 1.5], alpha=0.3, colors=['#3498db', '#e74c3c'])
ax1.scatter(X_data[y_data == 0, 0], X_data[y_data == 0, 1], color='#2980b9', edgecolors='k', s=30)
ax1.scatter(X_data[y_data == 1, 0], X_data[y_data == 1, 1], color='#c0392b', edgecolors='k', s=30)
ax1.set_title("Cây quyết định đơn lẻ (Single Decision Tree)\n(Ranh giới trực giao bậc thang, dễ quá khớp)", fontsize=11, fontweight='bold')

ax2.contourf(xx, yy, Z_rf, levels=[-0.5, 0.5, 1.5], alpha=0.3, colors=['#3498db', '#e74c3c'])
ax2.scatter(X_data[y_data == 0, 0], X_data[y_data == 0, 1], color='#2980b9', edgecolors='k', s=30)
ax2.scatter(X_data[y_data == 1, 0], X_data[y_data == 1, 1], color='#c0392b', edgecolors='k', s=30)
ax2.set_title("Rừng ngẫu nhiên (Random Forest - 25 Cây)\n(Ranh giới mượt mà, triệt tiêu phương sai, khái quát hóa vượt trội)", fontsize=11, fontweight='bold')

plt.suptitle("Đối chiếu Ranh giới Quyết định: Cây Đơn lẻ vs Học Tổ hợp Random Forest", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Tổng kết bài học thực hành

1. **$k$-NN:** Trực quan hóa cách $k$ điều khiển sự cân bằng Bias-Variance; $k$ nhỏ ranh giới rất nhạy cảm với từng điểm dữ liệu, $k$ lớn ranh giới ổn định nhưng làm mờ phân định.
2. **Cây quyết định:** Chia cắt không gian thành các hình hộp chữ nhật trực giao; rất dễ giải thích nhưng cây sâu có phương sai cao.
3. **Random Forest:** Minh chứng sức mạnh của Bagging: Biểu quyết từ 25 cây độc lập giúp ranh giới quyết định trở nên mượt mà, bao quát đúng hình dáng bán nguyệt tiềm ẩn của dữ liệu mà không bị bẫy nhiễu cục bộ.